# Getting started

Based on this [collab notebook](https://colab.research.google.com/drive/12YpI99LkuFeWcuYHt_idl142DqX7AaJf).

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

In [ ]:
import torch
import matplotlib.pyplot as plt
from pfns.priors import Batch
import math
from collections.abc import Callable
from pfns.model import bar_distribution
from pfns.model.bar_distribution import BarDistributionConfig
from pfns.model.encoders import EncoderConfig
from pfns.priors.prior import AdhocPriorConfig
from pfns.train import (
    BatchShapeSamplerConfig,
    MainConfig,
    OptimizerConfig,
    TransformerConfig,
    train,
)
import sklearn.linear_model
from pfns.priors.fast_gp import get_batch as get_batch_for_gp
import polars as pl
from plotnine import aes, geom_line, geom_point, geom_ribbon, ggplot
from plotnine import scale_color_manual

SEED = 42
torch.manual_seed(SEED)

In [ ]:
def sample_from_prior(
    num_datasets:int,
    num_features:int,
    num_points_in_each_dataset:int,
    hyperparameters:dict[str,float],
):
    loc_ws = torch.zeros(num_features + 1)
    ws = torch.distributions.Normal(loc_ws, hyperparameters["b"]).sample(
        (num_datasets,)
    ) 

    xs = torch.rand(num_datasets, num_points_in_each_dataset, num_features)
    loc_ys = torch.einsum(
            "nmf, nf -> nm",
            torch.cat([xs, torch.ones(num_datasets, num_points_in_each_dataset, 1)], 2),
            ws,
        )
    ys = torch.distributions.Normal(
        loc_ys,
        hyperparameters["a"],
    ).sample()
    return xs, ys

# in our convention we name the `num_datasets` -> `batch_size`, and the `num_points_in_each_dataset` -> `seq_len`

def get_batch_for_ridge_regression(
    batch_size:int=2, seq_len:int=100, num_features:int=1, hyperparameters:dict[str,float] | None=None, device:str="cpu", **kwargs
):
    if hyperparameters is None:
        hyperparameters = {"a": 0.1, "b": 1.0}
    ws = torch.distributions.Normal(torch.zeros(num_features + 1), hyperparameters["b"]).sample(
        (batch_size,)
    )

    xs = torch.rand(batch_size, seq_len, num_features)
    ys = torch.distributions.Normal(
        torch.einsum("nmf, nf -> nm", torch.cat([xs, torch.ones(batch_size, seq_len, 1)], 2), ws),
        hyperparameters["a"],
    ).sample()[..., None]

    # get_batch functions return two different ys, let's come back to this later, though.
    return Batch(x=xs.to(device), y=ys.to(device), target_y=ys.to(device))

def get_batch_for_sine_ridge_regression(
    batch_size:int=10, seq_len:int=100, num_features:int=1, hyperparameters:dict[str,float]|None=None, **kwargs
):
    if hyperparameters is None:
        hyperparameters = {"a": 0.01, "b": 1.0, "s": 0.3}
    ws = torch.distributions.Normal(torch.zeros(num_features + 1), hyperparameters["b"]).sample(
        (batch_size,)
    )

    xs = torch.rand(batch_size, seq_len, num_features)
    ys = torch.distributions.Normal(
        torch.einsum("nmf, nf -> nm", torch.cat([xs, torch.ones(batch_size, seq_len, 1)], 2), ws),
        hyperparameters["a"],
    ).sample()

    ## the new part

    ys += torch.sin(xs[..., 0] * 20) * hyperparameters["s"]

    ## end new part

    # get_batch functions return two different ys, let's come back to this later, though.
    return Batch(x=xs, y=ys[..., None], target_y=ys[..., None])

def get_batch_for_anti_sine_ridge_regression(
    batch_size:int=10, seq_len:int=100, num_features:int=1, hyperparameters:dict[str,float]|None=None, **kwargs
):
    if hyperparameters is None:
        hyperparameters = {"a": 0.01, "b": 1.0, "s": 0.3}
    ws = torch.distributions.Normal(torch.zeros(num_features + 1), hyperparameters["b"]).sample(
        (batch_size,)
    )

    xs = torch.rand(batch_size, seq_len, num_features)
    mean_ys = torch.einsum(
        "nmf, nf -> nm", torch.cat([xs, torch.ones(batch_size, seq_len, 1)], 2), ws
    )
    ys = torch.distributions.Normal(mean_ys, hyperparameters["a"]).sample()

    ys += torch.sin(xs[..., 0] * 20) * hyperparameters["s"]

    # the new part is in target_y
    return Batch(x=xs, y=ys[..., None], target_y=mean_ys[..., None])

def train_a_pfn(
    device:str,
    get_batch_function:Callable[...,Batch],
    epochs:int,
    num_features:int,
    max_dataset_size:int,
    hps:dict[str,float]|None,
    batch_size:int=256,
    steps_per_epoch:int=100,
):

    # define a bar distribution (riemann distribution) criterion with 1000 bars
    ys = get_batch_function(100000, 20, num_features, hyperparameters=hps).target_y
    # we define our bar distribution adaptively with respect to the above sample of target ys from our prior
    borders = bar_distribution.get_bucket_borders(num_outputs=1_000, ys=ys).tolist()

    config = MainConfig(
        priors=[
            AdhocPriorConfig(
                get_batch_methods=[get_batch_function],
                prior_kwargs={"num_features": num_features, "hyperparameters": hps},
            )
        ],
        optimizer=OptimizerConfig("adamw", lr=0.0003),
        model=TransformerConfig(
            criterion=BarDistributionConfig(full_support=True, borders=borders),
            emsize=512,
            nhead=8,
            nhid=1024,
            nlayers=6,
            features_per_group=1,
            attention_between_features=False,
            # The encoder config ensures the uniform inputs between 0 and 1 have mean 0 and var 1
            encoder=EncoderConfig(
                constant_normalization_mean=0.5,
                constant_normalization_std=math.sqrt(1 / 12),
            ),
        ),
        batch_shape_sampler=BatchShapeSamplerConfig(
            batch_size=batch_size,
            max_seq_len=max_dataset_size,
            min_num_features=num_features,
            max_num_features=num_features,
        ),
        epochs=epochs,
        warmup_epochs=epochs // 4,
        steps_per_epoch=steps_per_epoch,
        num_workers=0,
    )

    train_result = train(config, device=device, reusable_config=False)
    return train_result

## PFN to do ridge regression

### The prior

Our prior has two parts. i) What the mapping from x to y looks like, that is
$$
f = x^Tw\\
y \sim Normal(f, a^2I).
$$

We first map to the function value without noise, then we add normal noise.
While $x$ is our input, we have two more undefined symbols. $a$ is the standard deviation of our outputs and is fixed upfront to some value, e.g. $0.1$. $w$ on the other hand controls the form of our function, in one d this could e.g. define whether we go up or down with increasing $x$.

As $w$ is so crucial, we define a prior over it, too. That is we say $w \sim Normal(0, b^2I)$ for some other fixed standard deviation $b$. BTW in Bayesian terms a variable like $w$ that controls what our distribution looks like is generally called latent.

The above is the whole prior of ridge regression. We define a distribution over $w$ and then given that $w$ we define a distribution over mappings from $x$ to $y$.


### Generating prior samples for PFN training

All we want to do is train a neural network on datasets sampled from a prior. To do this, we need to define one more thing, that is: we need to define a distribution over $x$ that we consider. For simplicity let us just consider the uniform distribution between 0 and 1.


### Let's generate some samples in 1D

In [ ]:
num_datasets = 10
num_points_in_each_dataset = 100

hps = None
num_features = 1
max_dataset_size = 20
device = "mps"

Plotting our sample datasets

In [ ]:
batch = get_batch_for_ridge_regression()

In [ ]:
for dataset_index in range(batch.x.shape[0]):
    plt.scatter(batch.x[dataset_index, :, 0].numpy(), batch.y[dataset_index].numpy())
    # plt.scatter(xs[dataset_index, :, 0].numpy(), ys[dataset_index].numpy())

## Let's train a mini PFN for 1D datasets with up to 20 elements 🚀

### What happens during training?
During training we sample datasets from our `get_batch` method, we then split them into left and right of the so called `single_eval_pos` (which in turn is sampled uniformly at random). If something is left it is training, on the right it is test. Now our PFN is training to predict the $y$ of the test set, given $x$ and $y$ for the training set, as well as $x$ for the test set.

Our training uses standard Adam and a cross-entropy loss. We use a transformer as PFN, where each example in the training or test set is encoded as a token (sometimes also called time step). More info on the architecture can be found in our paper.

In [ ]:
epochs = 10

train_result = train_a_pfn(device,get_batch_for_ridge_regression, epochs, num_features, max_dataset_size,hps=hps)

In [ ]:
trained_model = train_result["model"]

Inference over a new sample data set

In [ ]:
batch = get_batch_for_ridge_regression(seq_len=100, hyperparameters=hps, batch_size=10)

In [ ]:
# seq dimension first

batch_index = 0  # change this to see other examples
num_training_points = 4

train_x = batch.x[batch_index, :num_training_points]
train_y = batch.y[batch_index, :num_training_points]
test_x = batch.x[batch_index]

with torch.no_grad():
    # we add our batch dimension, as our transformer always expects that
    logits = trained_model(train_x[None], train_y[None], test_x[None])

# the model has the criterion still attached (it is the same though, as our criterion above)
# the criterion has a lot of handy function to use these logits
pred_means = trained_model.criterion.mean(logits)[0]
pred_confs = trained_model.criterion.quantile(logits)[0]

In [ ]:
train_df = pl.DataFrame({
    "x": train_x[..., 0].detach().cpu().numpy().reshape(-1),
    "y": train_y.detach().cpu().numpy().reshape(-1),
})

x_test = test_x[..., 0].detach().cpu().numpy().reshape(-1)
confs = pred_confs.detach().cpu().numpy()

pred_df_pfn = pl.DataFrame({
    "x": x_test,
    "mean": pred_means.detach().cpu().numpy().reshape(-1),
    "lower": confs[:, 0],
    "upper": confs[:, 1],
}).sort("x")

training a reference ridge model

In [ ]:
a = 0.1
b = 1.0

ridge_model = sklearn.linear_model.Ridge(alpha=(a / b) ** 2)

In [ ]:
ridge_model.fit(train_x, train_y)

comparing predictions and confidence intervals

In [ ]:
y_pred_ridge = ridge_model.predict(test_x.detach().cpu().numpy()).reshape(-1)

ridge_df = pl.DataFrame({
    "x": x_test,
    "y": y_pred_ridge,
}).with_columns(pl.lit("Ridge regression").alias("model"),pl.col("y").cast(pl.Float32))

In [ ]:
line_df = pl.concat([
    pred_df_pfn.select(
        "x",
        pl.col("mean").alias("y"),
    ).with_columns(pl.lit("PFN").alias("model")),
    ridge_df.select("x", "y", "model"),
]).sort("x")

In [ ]:
(
    ggplot()
    + geom_ribbon(
        data=pred_df_pfn,
        mapping=aes(x="x", ymin="lower", ymax="upper"),
        fill="green",
        color="none",
        alpha=0.1,
    )
    + geom_line(
        data=line_df,
        mapping=aes(x="x", y="y", color="model"),
    )
    + geom_point(
        data=train_df,
        mapping=aes(x="x", y="y"),
    )
    + scale_color_manual(
        values={"PFN": "green", "Ridge regression": "blue"},
        name=None,
    )
)

Our model uses a Riemann distribution, that is a discretized distribution internally. We can show this here for one prediction.

In [ ]:
trained_model.criterion.mode(logits)

In [ ]:
probs_of_first_pred = logits[0, order_test_x[0]].softmax(0)
borders = trained_model.criterion.borders
widths = borders[1:] - borders[:-1]
density_of_first_pred = probs_of_first_pred / widths

In [ ]:
plt.bar(borders[:-1].tolist(), density_of_first_pred.tolist(), width=widths, align="edge")

## Modelling other $y = f(x)$ beyond linear

### i) Sine Ridge Regression

In [ ]:
batch = get_batch_for_sine_ridge_regression()
for dataset_index in range(len(batch.x)):
    plt.scatter(batch.x[dataset_index, :, 0].numpy(), batch.y[dataset_index, :].numpy())

In [ ]:
epochs = 20

train_results_sine = train_a_pfn(device,get_batch_for_sine_ridge_regression, epochs, num_features, max_dataset_size, hps)
trained_model_sine = train_results_sine["model"]

In [ ]:
batch = get_batch_for_sine_ridge_regression(seq_len=300)

In [ ]:
# our model wants the seq dimension first, remember that!

batch_index = 2  # change this to see other examples
num_training_points = 6

train_x = batch.x[batch_index, :num_training_points]
train_y = batch.y[batch_index, :num_training_points]
test_x = batch.x[batch_index]

with torch.no_grad():
    # we add our batch dimension, as our transformer always expects that

    logits = trained_model_sine(train_x[None], train_y[None], test_x[None])
# the model has the criterion still attached (it is the same though, as our criterion above)
# the criterion has a lot of handy function to use these logits
pred_means = trained_model_sine.criterion.mean(logits)[0]
pred_confs = trained_model_sine.criterion.quantile(logits)[0]


plt.scatter(train_x[..., 0], train_y)
order_test_x = test_x[..., 0].argsort()
plt.plot(test_x[order_test_x], pred_means[order_test_x], color="green", label="pfn")
plt.fill_between(
    test_x[order_test_x, 0],
    pred_confs[..., 0][order_test_x],
    pred_confs[..., 1][order_test_x],
    alpha=0.1,
    color="green",
)

### ii) Anti-sine Regression

In all `get_batch` functions so far we specified two `y`s. We did this because there is a difference in how they are treated by the training loop. `batch.y` is used to feed it to the PFN as knowledge about the training set, while `batch.target_y` is used to train the transformer to predict it.

Typically we use a different `batch.target_y` s.t. we train not to fit the noise. In EI for example one often prefers EI over predictions that do not take inherent noise into account, as that changes the confidence intervals.

We will use the same prior as before, but make our model predict the middle of the sine wave instead of the wave itself.

Let's visualize this for one example:

In [ ]:
batch = get_batch_for_anti_sine_ridge_regression()

In [ ]:
plt.scatter(batch.x[dataset_index, :, 0].numpy(), batch.y[dataset_index].numpy(), label="y")
plt.scatter(
    batch.x[dataset_index, :, 0].numpy(), batch.target_y[dataset_index].numpy(), label="target_y"
)

plt.legend()
plt.show()

In [ ]:
epochs = 10

# we use a large max_dataset_size s.t. we can show it with more training examples, which makes the effect more clear

train_results_anti_sine = train_a_pfn(
    device,
    get_batch_for_anti_sine_ridge_regression, epochs, num_features, 100, hps, batch_size=64
)

In [ ]:
trained_model_anti_sine = train_results_anti_sine["model"]

If you see "nan" in the positional losses in the training log above that just means this particular `single_eval_pos` was not sampled.

In [ ]:
# our model wants the seq dimension first, remember that!

batch_index = 0  # change this to see other examples
num_training_points = 50

train_x = batch.x[batch_index, :num_training_points]
train_y = batch.y[batch_index, :num_training_points]
test_x = batch.x[batch_index]

with torch.no_grad():
    # we add our batch dimension, as our transformer always expects that
    logits = trained_model_anti_sine(train_x[None], train_y[None], test_x[None])

# the model has the criterion still attached (it is the same though, as our criterion above)
# the criterion has a lot of handy function to use these logits
pred_means = trained_model_anti_sine.criterion.mean(logits)[0]
pred_confs = trained_model_anti_sine.criterion.quantile(logits)[0]

In [ ]:
plt.scatter(train_x[..., 0], train_y)
order_test_x = test_x[..., 0].argsort()
plt.plot(test_x[order_test_x, ..., 0], pred_means[order_test_x], color="green", label="pfn")
plt.fill_between(
    test_x[order_test_x, ..., 0],
    pred_confs[..., 0][order_test_x],
    pred_confs[..., 1][order_test_x],
    alpha=0.1,
    color="green",
)

It looks like we can actually reconstruct the mean of the sine. This might be helpful for some obscure regression with a sine based noise..

### iii) Gaussian Process

Let's sample some datasets in 1d.

In [ ]:
batch = get_batch_for_gp(
    batch_size=2,
    seq_len=100,
    num_features=1,
    hyperparameters={"noise": 1e-4, "outputscale": 1.0, "lengthscale": 0.1},
)
for dataset_index in range(len(batch.x)):
    plt.scatter(batch.x[dataset_index, :, 0].cpu().numpy(), batch.y[dataset_index, :].cpu().numpy())

In [ ]:
epochs = 10

training_output_gp = train_a_pfn(
    device,
    get_batch_for_gp, epochs, num_features, max_dataset_size, hps={"noise": 1e-4, "outputscale": 1.0, "lengthscale": 0.1}
)

In [ ]:
trained_model_gp = training_output_gp["model"]

In [ ]:
# our model wants the seq dimension first, remember that!

batch = get_batch_for_gp(
    batch_size=2,
    seq_len=100,
    num_features=1,
    hyperparameters={"noise": 1e-4, "outputscale": 1.0, "lengthscale": 0.1},
)

In [ ]:
batch_index = 0  # change this to see other examples
num_training_points = 5

train_x = batch.x[batch_index, :num_training_points].cpu()
train_y = batch.y[batch_index, :num_training_points].cpu()
test_x = batch.x[batch_index].cpu()

with torch.no_grad():
    # we add our batch dimension, as our transformer always expects that
    logits = trained_model_gp(train_x[None], train_y[None], test_x[None])

In [ ]:
# the model has the criterion still attached (it is the same though, as our criterion above)
# the criterion has a lot of handy function to use these logits
pred_means = trained_model_gp.criterion.mean(logits)[0]
pred_confs = trained_model_gp.criterion.quantile(logits)[0]

In [ ]:
plt.scatter(train_x[..., 0], train_y)
order_test_x = test_x[..., 0].argsort()
plt.plot(test_x[order_test_x, ..., 0], pred_means[order_test_x], color="green", label="pfn")
plt.fill_between(
    test_x[order_test_x, ..., 0],
    pred_confs[..., 0][order_test_x],
    pred_confs[..., 1][order_test_x],
    alpha=0.1,
    color="green",
)

pred_ei = trained_model_gp.criterion.ei(logits, best_f=max(train_y))

plt.figure()
plt.title("EI")
plt.plot(test_x[order_test_x, ..., 0], pred_ei[0, order_test_x], color="green", label="ei")
plt.show()